In [1]:
import pandas as pd
import numpy as np

In [2]:
emails = pd.read_csv(
    "../../data/01_raw/raw_emails.csv",
    low_memory=False
)

emails.shape

(123389, 27)

In [3]:
emails.info()
emails.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123389 entries, 0 to 123388
Data columns (total 27 columns):
 #   Column                                Non-Null Count   Dtype 
---  ------                                --------------   ----- 
 0   Co_Ref                                123389 non-null  object
 1   Time_to_Renewal                       123389 non-null  object
 2   crm_accreditation_completed           102354 non-null  object
 3   crm_timely_completion                 102354 non-null  object
 4   crm_progress_towards_accreditation    102354 non-null  object
 5   crm_delays_in_accreditation           102354 non-null  object
 6   crm_contractor_suggested_leave        102354 non-null  object
 7   crm_contractor_engagement             102354 non-null  object
 8   crm_contractor_sentiment              102354 non-null  object
 9   crm_contractor_sentiment_score        102354 non-null  object
 10  crm_dts_or_ssip_mentioned             102354 non-null  object
 11  crm_customer_

,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


In [5]:
emails = emails.drop_duplicates()
print("Shape after duplicate removal:", emails.shape)

Shape after duplicate removal: (123389, 27)


In [6]:
emails.columns = (
    emails.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [7]:
obj_cols = emails.select_dtypes(include="object").columns

for col in obj_cols:
    emails[col] = emails[col].astype(str).str.strip()

In [8]:
date_cols = [
    "email_date",
    "sent_date",
    "opened_date",
    "clicked_date"
]

for col in date_cols:
    if col in emails.columns:
        emails[col] = pd.to_datetime(
            emails[col],
            errors="coerce"
        )

In [9]:
possible_numeric_cols = [
    "open_count",
    "click_count",
    "bounce_count",
    "engagement_score"
]

for col in possible_numeric_cols:
    if col in emails.columns:
        emails[col] = pd.to_numeric(
            emails[col],
            errors="coerce"
        )

In [10]:
for col in obj_cols:
    emails[col] = emails[col].replace({
        "opened": "Opened",
        "OPENED": "Opened",
        "clicked": "Clicked",
        "CLICKED": "Clicked",
        "sent": "Sent",
        "SENT": "Sent"
    })

In [11]:
cat_cols = emails.select_dtypes(include="object").columns

for col in cat_cols:
    emails[col] = emails[col].replace("nan", np.nan)
    emails[col] = emails[col].fillna("Unknown")

In [12]:
num_cols = emails.select_dtypes(include=np.number).columns

for col in num_cols:
    emails[col] = emails[col].fillna(
        emails[col].median()
    )

In [13]:
emails.isnull().sum().sort_values(ascending=False).head(20)

co_ref                                  0
crm_platform_issues_raised              0
crm_financial_hardship_mentioned        0
crm_dissatisfaction_with_support        0
crm_negative_customer_experience        0
crm_refund_mentioned                    0
crm_customer_complained                 0
crm_dissatisified_with_renewal_price    0
crm_auto_renewal_status                 0
crm_membership_overdue                  0
crm_accreditation_issues                0
crm_agent_chase_count                   0
crm_agent_chased_contractor             0
crm_membership_level                    0
time_to_renewal                         0
crm_competitors_mentioned               0
crm_customer_payment_intention          0
crm_dts_or_ssip_mentioned               0
crm_contractor_sentiment_score          0
crm_contractor_sentiment                0
dtype: int64

In [15]:
emails.to_csv(
    "../../data/02_processed/processed_emails.csv",
    index=False
)

In [16]:
processed_emails = pd.read_csv(
    "../../data/02_processed/processed_emails.csv"
)

processed_emails.shape

C:\Users\shrey\AppData\Local\Temp\ipykernel_16884\2023363279.py:1: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  processed_emails = pd.read_csv(


(123389, 27)

In [17]:

# 🔍 COMPREHENSIVE EMAIL DATA QUALITY ANALYSIS

print("="*70)
print("EMAIL DATA: CLEANING & COLUMN ANALYSIS")
print("="*70)

print(f"\nDataset Shape: {processed_emails.shape}")
print(f"Total nulls: {processed_emails.isnull().sum().sum()}\n")

# 1. Check ID/Tracking columns
print("1️⃣  ID/TRACKING COLUMNS (cardinality check):")
id_keywords = ["id", "ref", "email_id", "unnamed", "row"]
for col in processed_emails.columns:
    if any(keyword in col.lower() for keyword in id_keywords):
        unique_pct = processed_emails[col].nunique() / len(processed_emails) * 100
        print(f"   {col}: {processed_emails[col].nunique()} unique ({unique_pct:.1f}%)")

# 2. Check for completely empty/sparse columns
print("\n2️⃣  EMPTY/SPARSE COLUMNS (>80% missing):")
sparse_found = False
for col in processed_emails.columns:
    missing_pct = processed_emails[col].isnull().sum() / len(processed_emails) * 100
    if missing_pct > 80:
        print(f"   {col}: {missing_pct:.1f}% missing")
        sparse_found = True
if not sparse_found:
    print("   ✅ None found")

# 3. Check low-variance columns (>90% one value)
print("\n3️⃣  LOW-VARIANCE COLUMNS (>90% single value):")
low_var_found = False
for col in processed_emails.select_dtypes(include="object").columns:
    top_pct = processed_emails[col].value_counts().head(1).values[0] / len(processed_emails) * 100
    if top_pct > 90:
        top_val = processed_emails[col].value_counts().head(1).index[0]
        print(f"   {col}: '{top_val}' = {top_pct:.1f}%")
        low_var_found = True
if not low_var_found:
    print("   ✅ None found")

# 4. Check "Unknown" concentration
print("\n4️⃣  'UNKNOWN' VALUE CONCENTRATION:")
unknown_cols = []
for col in processed_emails.select_dtypes(include="object").columns:
    unknown_pct = (processed_emails[col] == "Unknown").sum() / len(processed_emails) * 100
    if unknown_pct > 30:
        print(f"   {col}: {unknown_pct:.1f}% Unknown")
        unknown_cols.append(col)

# 5. Check date column NaT rates
print("\n5️⃣  DATE COLUMNS - NaT Percentage:")
for col in processed_emails.select_dtypes(include=['datetime64']).columns:
    nat_pct = processed_emails[col].isna().sum() / len(processed_emails) * 100
    print(f"   {col}: {nat_pct:.1f}% NaT")

# 6. Check numeric columns
print("\n6️⃣  NUMERIC COLUMNS - Value Distribution:")
for col in processed_emails.select_dtypes(include=[np.number]).columns:
    zero_count = (processed_emails[col] == 0).sum()
    zero_pct = zero_count / len(processed_emails) * 100
    print(f"   {col}: min={processed_emails[col].min():.2f}, max={processed_emails[col].max():.2f}, zeros={zero_pct:.1f}%")

# 7. Column type summary
print("\n7️⃣  COLUMN TYPE DISTRIBUTION:")
print(f"   Categorical (object): {processed_emails.select_dtypes(include='object').shape[1]}")
print(f"   Numeric (int/float): {processed_emails.select_dtypes(include=[np.number]).shape[1]}")
print(f"   Datetime: {processed_emails.select_dtypes(include=['datetime64']).shape[1]}")

print("\n" + "="*70)

EMAIL DATA: CLEANING & COLUMN ANALYSIS

Dataset Shape: (123389, 27)
Total nulls: 0

1️⃣  ID/TRACKING COLUMNS (cardinality check):
   co_ref: 37964 unique (30.8%)
   crm_refund_mentioned: 28 unique (0.0%)

2️⃣  EMPTY/SPARSE COLUMNS (>80% missing):
   ✅ None found

3️⃣  LOW-VARIANCE COLUMNS (>90% single value):
   ✅ None found

4️⃣  'UNKNOWN' VALUE CONCENTRATION:

5️⃣  DATE COLUMNS - NaT Percentage:

6️⃣  NUMERIC COLUMNS - Value Distribution:
   year: min=2024.00, max=2026.00, zeros=0.0%

7️⃣  COLUMN TYPE DISTRIBUTION:
   Categorical (object): 26
   Numeric (int/float): 1
   Datetime: 0



In [18]:

# 🧹 EMAIL DATA CLEANING & OPTIMIZATION

print("\n📋 PERFORMING EMAIL DATA CLEANING...\n")

# 1. Drop ID columns with very high cardinality (>30% unique = mostly unique)
cols_to_drop = []
for col in processed_emails.columns:
    unique_pct = processed_emails[col].nunique() / len(processed_emails) * 100
    if col.lower() in ["co_ref", "customer_ref", "email_id", "reference_id"] and unique_pct > 30:
        cols_to_drop.append(col)
        print(f"   ✅ Marked '{col}' for dropping (high cardinality: {unique_pct:.1f}%)")

if cols_to_drop:
    processed_emails = processed_emails.drop(columns=cols_to_drop, errors='ignore')
    print(f"   → Dropped {len(cols_to_drop)} high-cardinality ID columns")

# 2. Detect and handle text/paragraph columns (free text that won't help modeling)
print(f"\n   Analyzing text columns for free-text data...")
text_cols_to_convert = []
for col in processed_emails.select_dtypes(include="object").columns:
    if col not in ["co_ref", "year"]:  # Skip reference columns
        sample = processed_emails[col].dropna().head(100)
        avg_length = sample.astype(str).str.len().mean()
        unique_count = processed_emails[col].nunique()
        
        # If average string length > 50 AND high unique count, it's likely free text
        if avg_length > 50 and unique_count > len(processed_emails) * 0.1:
            text_cols_to_convert.append(col)
            print(f"   ⚠️  '{col}': avg_len={avg_length:.0f}, unique={unique_count} (free text detected)")

# 3. Convert yes/no/true/false columns to binary
print(f"\n   Converting binary columns to numeric...")
binary_converted = 0
for col in processed_emails.select_dtypes(include="object").columns:
    if col not in text_cols_to_convert:
        unique_vals = set(processed_emails[col].dropna().unique())
        if unique_vals.issubset({'Yes', 'No', 'true', 'false', 'True', 'False', '1', '0'}):
            processed_emails[col] = processed_emails[col].replace({
                'Yes': 1, 'yes': 1, 'true': 1, 'True': 1, '1': 1,
                'No': 0, 'no': 0, 'false': 0, 'False': 0, '0': 0
            }).astype('int64')
            binary_converted += 1

if binary_converted > 0:
    print(f"   ✅ Converted {binary_converted} binary columns to numeric")

# 4. Final null check
null_check = processed_emails.isnull().sum().sum()
print(f"\n   Final null count: {null_check}")

# 5. Summary
print(f"\n✅ CLEANED EMAIL DATA:")
print(f"   Shape: {processed_emails.shape}")
print(f"   Columns removed: {len(cols_to_drop)}")
print(f"   Binary columns converted: {binary_converted}")
print(f"   Memory: {processed_emails.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 6. Save cleaned data
processed_emails.to_csv(
    "../../data/02_processed/processed_emails.csv",
    index=False
)
print(f"\n   ✅ Saved to processed_emails.csv")


📋 PERFORMING EMAIL DATA CLEANING...

   ✅ Marked 'co_ref' for dropping (high cardinality: 30.8%)
   → Dropped 1 high-cardinality ID columns

   Analyzing text columns for free-text data...

   Converting binary columns to numeric...

   Final null count: 0

✅ CLEANED EMAIL DATA:
   Shape: (123389, 26)
   Columns removed: 1
   Binary columns converted: 0
   Memory: 161.27 MB

   ✅ Saved to processed_emails.csv


In [19]:

# 📊 FINAL VERIFICATION & SUMMARY

print("\n📊 FINAL EMAIL DATA STATUS:\n")

# Reload the saved data
final_emails = pd.read_csv("../../data/02_processed/processed_emails.csv")

print(f"Shape: {final_emails.shape}")
print(f"Nulls: {final_emails.isnull().sum().sum()}")
print(f"Memory: {final_emails.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nColumn types:")
col_types = final_emails.dtypes.value_counts()
for dtype, count in col_types.items():
    print(f"   {dtype}: {count}")

print(f"\nColumn details:")
for col in final_emails.columns:
    dtype = final_emails[col].dtype
    unique = final_emails[col].nunique()
    
    if dtype == 'object':
        print(f"   {col} (categorical): {unique} unique values")
    elif dtype in ['int64', 'float64']:
        print(f"   {col} (numeric): min={final_emails[col].min()}, max={final_emails[col].max()}")
    else:
        print(f"   {col} ({dtype}): {unique} unique")

print(f"\n✅ Email data is clean and ready for modeling!")


📊 FINAL EMAIL DATA STATUS:



C:\Users\shrey\AppData\Local\Temp\ipykernel_16884\3280568794.py:6: DtypeWarning: Columns (15,18) have mixed types. Specify dtype option on import or set low_memory=False.
  final_emails = pd.read_csv("../../data/02_processed/processed_emails.csv")


Shape: (123389, 26)
Nulls: 0
Memory: 161.27 MB

Column types:
   object: 25
   int64: 1

Column details:
   time_to_renewal (categorical): 4 unique values
   crm_accreditation_completed (categorical): 4 unique values
   crm_timely_completion (categorical): 4 unique values
   crm_progress_towards_accreditation (categorical): 4 unique values
   crm_delays_in_accreditation (categorical): 19 unique values
   crm_contractor_suggested_leave (categorical): 4 unique values
   crm_contractor_engagement (categorical): 29 unique values
   crm_contractor_sentiment (categorical): 13 unique values
   crm_contractor_sentiment_score (categorical): 53 unique values
   crm_dts_or_ssip_mentioned (categorical): 10 unique values
   crm_customer_payment_intention (categorical): 7 unique values
   crm_competitors_mentioned (categorical): 6 unique values
   crm_membership_level (categorical): 58 unique values
   crm_platform_issues_raised (categorical): 4 unique values
   crm_agent_chased_contractor (categori